# FaceJPEG — اختبار صور جديدة بنموذج محفوظ
هذا دفتر للتنبؤ فقط من أربع خلايا؛ لا ينزل بيانات الوجوه ولا يعيد التدريب. يعمل على CPU.

**قبل مغادرة جلسة التدريب:** احفظ `best.pt` الناتج عن تدريب صور الوجوه على جهازك أو Drive. في دفتر التدريب يظهر مساره في المرحلة9، ويمكن تنزيله من لوحة Files أو استخراج ملفه من ZIP النتائج. حفظ الدفتر وحده لا يحفظ الأوزان. `last.pt` للاستكمال وليس بديلًا عن `best.pt` المختار بحدود التحقق.

شغّل الخلايا1–4 بالترتيب، وارفع ملف النموذج في الخلية3، ثم الصور من الواجهة في الخلية4. لا تحتاج حساب Kaggle أو تركيب Drive؛ يمكنك رفع النموذج من جهازك. أوزان QUICK الهندسية مرفوضة هنا لأنها لا تكشف أصالة الوجوه. لا توجد أوزان كشف وجوه حقيقية مضمّنة.

إغلاق تبويب Colab قد يُبقي الجلسة مدة، لكن الجلسة ورابط Gradio مؤقتان. إذا انتهت الجلسة أعد هذه الخلايا بملف النموذج المحفوظ، وافتح رابط الواجهة الجديد. ما دامت الواجهة تعمل اختبر صورًا متتابعة دون تشغيل الخلايا مجددًا. الدرجة ترجيح غير معاير وليست إثبات أصالة.

## 1 — تهيئة الكود
يفك الكود المضمن ويعرّف IN_COLAB والمسارات؛ شغّله أولًا في كل جلسة جديدة.

In [ ]:
import os, sys, json, gzip, base64, hashlib, importlib.util, subprocess, time
from pathlib import Path
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
_blob = base64.b64decode('...')
assert hashlib.sha256(_blob).hexdigest() == '8f6a543642c2f7837961cc7e0a6e7c261e7de5c3b5dba4a06c49064f17ac845b', "Embedded source checksum failed"
_files = json.loads(gzip.decompress(_blob).decode("utf-8"))
_base = Path(os.environ.get("FACEJPEG_WORKSPACE", "/content" if IN_COLAB else str(Path.cwd())))
PROJECT = (_base / "facejpeg_final_v2_2").resolve()
PROJECT.mkdir(parents=True, exist_ok=True)
for _rel, _text in _files.items():
    _path = (PROJECT / _rel).resolve()
    if not _path.is_relative_to(PROJECT):
        raise ValueError("Unsafe source path")
    if _path.exists() and _path.read_text(encoding="utf-8") != _text:
        raise FileExistsError(f"{_path} has local changes: choose a new FACEJPEG_WORKSPACE or retain those changes")
    _path.parent.mkdir(parents=True, exist_ok=True)
    _path.write_text(_text, encoding="utf-8")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT)) if str(PROJECT) not in sys.path else None
def _refresh_project_imports():
    # Reload only this project's modules if an earlier notebook used another folder.
    removed = []
    for _name, _module in list(sys.modules.items()):
        if _name == "facejpeg" or _name.startswith("facejpeg.") or _name in {"app", "scripts.make_demo", "scripts.download_data"}:
            _file = getattr(_module, "__file__", None)
            if not _file or not Path(_file).resolve().is_relative_to(PROJECT):
                sys.modules.pop(_name, None)
                removed.append(_name)
    return removed
_removed = _refresh_project_imports()
if _removed:
    print("أعيد تهيئة وحدات المشروع من النسخة الجديدة:", len(_removed))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT / ".cache/matplotlib"))
_torch_pair_ok = False
try:
    import torch, torchvision
    _torch_pair_ok = True
    print("PyTorch:", torch.__version__, "Torchvision:", torchvision.__version__)
    print("CUDA build:", torch.version.cuda, "GPU available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        _free, _total = torch.cuda.mem_get_info()
        print("GPU:", torch.cuda.get_device_name(), "free/total GiB:", round(_free/2**30,2), round(_total/2**30,2))
    else:
        print("التنبؤ متاح على CPU؛ لا يحتاج GPU أو بيانات التدريب.")
except ImportError:
    print("Torch/Torchvision غير مثبتين؛ ستُعرض تعليمات التثبيت في المرحلة2.")
except Exception as _error:
    print("فشل استيراد زوج Torch/Torchvision:", type(_error).__name__)
print("Project:", PROJECT)

## 2 — تثبيت متطلبات التنبؤ
يحافظ على زوج Torch/Torchvision في Colab ويثبت متطلبات الواجهة فقط. لا ينزل مجموعة صور أو أوزان ImageNet.

In [ ]:
import threading
if IN_COLAB and not _torch_pair_ok:
    raise RuntimeError("ابدأ جلسة Colab نظيفة مع CPU أو GPU حسب النمط وأعد التشغيل؛ لن نستبدل زوج Torch المعطوب تلقائيًا.")
if _torch_pair_ok:
    import re
    _torch_version = tuple(map(int, re.match(r"([0-9]+)[.]([0-9]+)", str(torch.__version__)).groups()))
    if not (2,5) <= _torch_version < (3,0):
        raise RuntimeError("يلزم Torch >=2.5 و<3 مع Torchvision المتوافق؛ اختر إصدارًا مدعومًا من تعليمات PyTorch الرسمية أو جلسة Colab حديثة.")
_req = "requirements-predict-colab.txt" if _torch_pair_ok else "requirements.txt"
_done = threading.Event()
_install_start = time.monotonic()
def _install_heartbeat():
    while not _done.wait(15):
        print(f"[تثبيت المتطلبات] ما زالت عملية pip تعمل؛ elapsed={time.monotonic()-_install_start:.0f}s — راجع آخر عملية تنزيل أعلاه", flush=True)
_thread = threading.Thread(target=_install_heartbeat, daemon=True)
_thread.start()
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", _req], check=True)
finally:
    _done.set(); _thread.join(timeout=1)
import torch, torchvision
from facejpeg.runtime import hardware_info
from IPython.display import display, Markdown, Image as DisplayImage
print("جاهز لتحميل best.pt؛ التنبؤ لا يحتاج GPU.")
print(f"انتهى التثبيت والتحقق خلال {time.monotonic()-_install_start:.1f}s")

## 3 — تحميل النموذج المحفوظ
في Colab اترك CHECKPOINT_PATH فارغًا واختر ملف `best.pt` من جهازك عند ظهور زر الرفع. أو اكتب مسار ملف موجود في الجلسة. محليًا اكتب مساره. يُفحص ترتيب الفئات والمعالجة وحدود القرار المحفوظة. لا تغيّر الحدود ولا تحتاج بيانات التدريب/الاختبار للتنبؤ.

In [ ]:
MODEL_READY = False
PREDICT_CHECKPOINT = None
CHECKPOINT_PATH = ""  # مثال /content/best.pt؛ اتركه فارغًا للرفع من جهازك
from facejpeg.infer import Predictor
if not CHECKPOINT_PATH.strip():
    if not IN_COLAB:
        raise ValueError("اكتب مسار best.pt في CHECKPOINT_PATH ثم أعد هذه الخلية.")
    from google.colab import files
    _uploaded = files.upload()
    if len(_uploaded) != 1:
        raise ValueError("اختر ملف نموذج واحدًا: best.pt الناتج عن تدريب صور الوجوه.")
    CHECKPOINT_PATH = next(iter(_uploaded))
if not Path(CHECKPOINT_PATH).is_file():
    raise FileNotFoundError("لم أجد ملف النموذج. ارفعه أو صحّح CHECKPOINT_PATH.")
_checked_predictor = Predictor(CHECKPOINT_PATH, device="cpu")
if not _checked_predictor.face_classification_enabled:
    del _checked_predictor
    raise ValueError("هذا نموذج اختبار أشكال QUICK؛ لا يكشف أصالة الوجوه. اختر best.pt بعد تدريب REAL_CPU أو REAL على بيانات الوجوه. تغيير الاسم أو MODE لا يكفي.")
PREDICT_CHECKPOINT = Path(CHECKPOINT_PATH).resolve()
MODEL_READY = True
print("تم تحميل نموذج تجربة الوجوه:", PREDICT_CHECKPOINT.name)
print("حدود القرار المحفوظة:", _checked_predictor.bundle["thresholds"])
del _checked_predictor

## 4 — فتح الواجهة واختبار الصور
ارفع JPG/PNG لوجه واحد، واختر جودة JPEG ثم اضغط التحليل. لتجربة صورة أخرى بدّل الصورة واضغط الزر؛ لا تعِد التدريب أو تحميل النموذج. في Colab يظهر رابط عام مؤقت؛ لا ترفع صورًا حساسة. إذا انتهت الجلسة افتح هذا الدفتر مجددًا وشغّل1–4 بملف best.pt نفسه.

In [ ]:
if not globals().get("MODEL_READY", False) or not globals().get("PREDICT_CHECKPOINT"):
    raise RuntimeError("شغّل الخلايا1–3 وحمّل best.pt الصالح أولًا.")
from app import create_app
if "demo" in globals():
    demo.close()
demo = create_app(PREDICT_CHECKPOINT)
demo.launch(share=IN_COLAB, debug=False)